# Kaggle: train, evaluate, and push to the ABSA model registry

**This is a template, not a runnable-as-is notebook.** It needs a GPU and
Kaggle context this repo's own dev environment doesn't have, plus two
things only you can fill in before it will do anything real:

1. **`HF_REPO_ID`** below — a Hugging Face Hub model repo you control
   (e.g. `"your-username/nlp-transformer-analysis-absa"`). The trained
   checkpoint gets pushed there; Neon only stores a pointer to it.
2. **An `HF_TOKEN` Kaggle secret** — Kaggle notebook Settings → Add-ons →
   Secrets → add a secret named `HF_TOKEN` with write access to that repo
   (from https://huggingface.co/settings/tokens).

## What this notebook does

Trains the main-cohort model (`src/train.py`'s `Trainer`, unchanged from
local runs) with **class-weighted loss** to correct for Amazon reviews'
positive skew (see step 3 — the first real run pushed to the registry
mostly predicted "positive" regardless of input, which is what this
fixes), evaluates it (`src/evaluate.py`'s `evaluate_model`), runs it over
the same curated example set `app.py` shows in its "Example Gallery"
page, pushes the checkpoint to Hugging Face Hub, and writes a
`results.json` output artifact matching the contract
`scripts/load_absa_results.py` expects.

## After this notebook finishes

1. Download `results.json` from this notebook's Output panel.
2. Locally (where your `.env` has `DATABASE_URL`), run:
   ```bash
   python scripts/load_absa_results.py results.json
   ```
3. `streamlit run app.py` — the sidebar should now show
   `Loaded trained run ...` instead of the untrained-baseline warning.

## 1. Setup

Kaggle's Python images already have `torch`/`transformers`/`datasets`
preinstalled. `datasets` versions newer than what `McAuley-Lab/Amazon-
Reviews-2023` supports will hit the same loading-script error this repo's
own `src/data.py` already runs into locally — if that happens here, pin
an older `datasets` version, or point `config.yaml`'s `data.dataset_name`
/`dataset_subset` at a dataset that doesn't need `trust_remote_code`.

In [15]:
!pip install -q huggingface_hub
# pip install src
!pip install -q "datasets<4"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.


In [16]:
import datasets

print(datasets.__version__)


5.0.0


In [17]:
import sys
from pathlib import Path

REPO_ROOT = Path("/notebooks")

if not (REPO_ROOT / "src").exists():
    raise FileNotFoundError(f"Missing src directory: {REPO_ROOT}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Using repository: {REPO_ROOT}")


Using repository: /notebooks


In [18]:
import yaml

with open(REPO_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)

from src.data import (
    create_dataloaders,
    compute_class_weights
)


## 2. Bring in this repo

Either upload `src/`, `config.yaml`, and `db/schema.sql` as a Kaggle
Dataset attached to this notebook, or clone the repo directly if it's
reachable from Kaggle (public, or private with a token):

In [19]:
from src.data import (
    create_dataloaders,
    compute_class_weights
)


In [20]:
dataloaders = create_dataloaders(config)

print(dataloaders.keys())


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Streaming load failed (Dataset scripts are no longer supported, but found Amazon-Reviews-2023.py), trying non-streaming...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


RuntimeError: Dataset scripts are no longer supported, but found Amazon-Reviews-2023.py

In [ ]:
train_dataset = dataloaders["train"].dataset

aspect_counts = {
    label: train_dataset.aspect_labels.count(label)
    for label in sorted(set(train_dataset.aspect_labels))
}

sentiment_counts = {
    label: train_dataset.sentiment_labels.count(label)
    for label in sorted(set(train_dataset.sentiment_labels))
}

print("Raw aspect counts:", aspect_counts)
print("Raw sentiment counts:", sentiment_counts)

print(
    "Aspect label counts:",
    {
        config["aspects"][label]: count
        for label, count in aspect_counts.items()
    }
)

print(
    "Sentiment label counts:",
    {
        config["sentiments"][label]: count
        for label, count in sentiment_counts.items()
    }
)


In [ ]:
from src.data import load_config, create_dataloaders, compute_class_weights
from src.model import build_model
from src.train import Trainer
from src.evaluate import evaluate_model
from src.inference import Predictor

config = load_config(str(REPO_ROOT / "config.yaml"))
# Optionally override for this run, e.g. a larger subset now that GPU time is available:
# config["data"]["train_size"] = 20000
# config["training"]["epochs"] = 5
config

## 3. Train

**Class-weighted loss.** Amazon reviews skew heavily positive (most
ratings are 4-5 stars), so `stars_to_sentiment`'s label distribution is
naturally imbalanced — training without correcting for this collapses
into mostly predicting "positive" regardless of input (this is exactly
what happened on the first real run pushed to the registry: `sentiment`
was "positive" in 5 of 7 example predictions, including on clearly
negative reviews). `compute_class_weights` (src/data.py) applies
sklearn's standard inverse-frequency "balanced" heuristic to both heads,
and the cell below prints the label counts so the skew — and the
resulting weights — are visible *before* spending GPU time on it.

**Dampened weighting.** Full inverse-frequency ("balanced") weighting is
itself prone to overcorrecting under severe imbalance and limited
training — an undertrained model can take the "cheap win" of collapsing
predictions onto whichever class currently has the highest loss-weight
(this is what happened after the fix above was first applied: predictions
collapsed toward "neutral", the rarest sentiment class, instead of
"positive"). `config.yaml`'s `training.class_weight_power` (default
`0.5`, sqrt-dampened) softens this; the cell below prints the resulting
weights so the effect is visible before training.

**Aspect keyword matching.** `assign_aspect` (src/data.py) now uses
word-boundary regex matching instead of naive substring matching, since
short shipping keywords like "late"/"box" were false-positive-matching
inside unrelated words ("chocolate", "boxed") common in the default
`raw_review_All_Beauty` subset, mechanically inflating shipping's share
of the labels regardless of true topic. The aspect label counts below
should look less shipping-skewed than a prior run.

In [ ]:
dataloaders = create_dataloaders(config)

train_dataset = dataloaders["train"].dataset
aspect_counts = {a: train_dataset.aspect_labels.count(a) for a in sorted(set(train_dataset.aspect_labels))}
sentiment_counts = {s: train_dataset.sentiment_labels.count(s) for s in sorted(set(train_dataset.sentiment_labels))}
print("Aspect label counts:   ", {config["aspects"][k]: v for k, v in aspect_counts.items()})
print("Sentiment label counts:", {config["sentiments"][k]: v for k, v in sentiment_counts.items()})

power = config["training"].get("class_weight_power", 1.0)
aspect_class_weights = compute_class_weights(
    train_dataset.aspect_labels, config["model"]["num_aspect_labels"], power=power
)
sentiment_class_weights = compute_class_weights(
    train_dataset.sentiment_labels, config["model"]["num_sentiment_labels"], power=power
)
print(f"Class weight power: {power}")
print("Aspect class weights:   ", aspect_class_weights.tolist())
print("Sentiment class weights:", sentiment_class_weights.tolist())

In [ ]:
model = build_model(
    config,
    aspect_class_weights=aspect_class_weights,
    sentiment_class_weights=sentiment_class_weights,
)
trainer = Trainer(model, config, dataloaders["train"], dataloaders["val"])
history = trainer.train()
history["train_loss"][-1], history["val_loss"][-1]

## 4. Evaluate

`metrics` below is copied verbatim into the results JSON pushed to Neon,
so there's no separate metric format to maintain. Then check it against
`config.yaml`'s `monitoring.*` success-criteria thresholds — see that
section's comment for what "floor" vs "target" mean and why they're
project-specific numbers, not generic ABSA benchmarks. This is the real
check for whether class-weighted loss actually fixed the positivity
skew: watch the negative/neutral recall rows specifically, not just
overall accuracy.

In [ ]:
checkpoint_path = str(Path(config["paths"]["model_dir"]) / "checkpoint_best.pt")
predictor = Predictor.from_checkpoint(checkpoint_path, config_path=str(REPO_ROOT / "config.yaml"))
metrics = evaluate_model(predictor, dataloaders["test"], config)
metrics["aspect"]["accuracy"], metrics["sentiment"]["accuracy"]

In [ ]:
from src.evaluate import check_targets, print_targets_report, print_report

print_report(metrics)
print_targets_report(check_targets(metrics, config))

## 5. Run the example set

Duplicated from `app.py`'s `EXAMPLE_REVIEWS` rather than imported — this
kernel has no reason to install `streamlit`/`altair` just to reuse a
seven-item dict. Keep this in sync with `app.py` by hand if the examples
there change.

In [ ]:
EXAMPLE_REVIEWS = {
    "[Mixed] Great quality, slow shipping": "Great quality but shipping took 3 weeks",
    "[Mixed] Food vs. service": "The food was excellent but the waiter was rude and slow.",
    "[Positive] Glowing review": "Absolutely love this product! Great quality and fast shipping.",
    "[Negative] Broke fast": "Terrible experience. Broke after one week. Customer service was unhelpful.",
    "[Neutral] Middling": "Decent for the price. Nothing special but gets the job done.",
    "[Negative] Damaged shipment": "Package arrived damaged. Took 3 weeks to get here. Very disappointed.",
    "[Positive] Easy setup": "Easy to set up and use. Instructions were clear. Good value for money.",
}

example_keys = list(EXAMPLE_REVIEWS.keys())
example_results = predictor.predict_batch(list(EXAMPLE_REVIEWS.values()))
example_predictions = [
    {
        "example_key": key,
        "text": r["text"],
        "aspect": r["aspect"],
        "aspect_confidence": r["aspect_confidence"],
        "sentiment": r["sentiment"],
        "sentiment_confidence": r["sentiment_confidence"],
    }
    for key, r in zip(example_keys, example_results)
]
example_predictions[:2]

## 6. Push the checkpoint to Hugging Face Hub

**Fill in `HF_REPO_ID` before running this cell.**

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

HF_REPO_ID = "your-username/nlp-transformer-analysis-absa"  # <-- fill in

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
commit_info = api.upload_file(
    path_or_fileobj=checkpoint_path,
    path_in_repo="checkpoint_best.pt",
    repo_id=HF_REPO_ID,
)
hf_revision = commit_info.oid
hf_revision

## 7. Assemble `results.json`

This is the exact contract `scripts/load_absa_results.py` validates
against — see that file's `REQUIRED_KEYS`.

In [ ]:
import json
from datetime import datetime, timezone

model_version = f"bert-absa-{datetime.now(timezone.utc):%Y-%m-%d}-kaggle-gpu-01"  # bump the suffix per run

hyperparameters = dict(config["training"])
hyperparameters["class_weighted_loss"] = True
hyperparameters["aspect_class_weights"] = aspect_class_weights.tolist()
hyperparameters["sentiment_class_weights"] = sentiment_class_weights.tolist()

results = {
    "model_version": model_version,
    "track": "bert-absa",
    "run_timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "environment": "kaggle_gpu",
    "hf_repo_id": HF_REPO_ID,
    "hf_revision": hf_revision,
    "hf_filename": "checkpoint_best.pt",
    "data_config": config["data"],
    "hyperparameters": hyperparameters,
    "metrics": metrics,
    "example_predictions": example_predictions,
    "notes": (
        f"Class-weighted loss (sklearn 'balanced'). Aspect counts: {aspect_counts}. "
        f"Sentiment counts: {sentiment_counts}."
    ),
}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Wrote /kaggle/working/results.json for {model_version}")

## 8. Next step

Download `/kaggle/working/results.json` from this notebook's Output
panel, then locally:

```bash
python scripts/load_absa_results.py results.json
```